In [1]:
# vfl_shap_party_level_ieee_kernel_agentic.py
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
import shap  # pip install shap

from vfl_utils import load_agent_definitions, split_features_by_agent_definitions
# -----------------------------


In [ ]:
# 1. Load CSV Dataset
# -----------------------------
#df = pd.read_csv("CICIDS2017_train.csv")  # update path if needed

df=pd.read_csv("datasets/undersampled_CIC2017_dataset.csv")

# Drop non-feature columns if needed (e.g., index, IDs)
df = df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")

# -----------------------------

In [3]:
# 2. Define Vertical Feature Split (from agentic_features.json)
# -----------------------------
# Use RAN / Edge / Core agents; only numeric columns present in the dataset are used.
non_feature_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "label", "label_numeric", "label_simplified"]
all_features = [col for col in df.columns if col not in non_feature_cols]
all_features = [col for col in all_features if df[col].dtype in ['int64', 'float64', 'int32', 'float32']]

AGENTIC_JSON = Path("agentic_features.json")
agent_definitions = load_agent_definitions(AGENTIC_JSON)
agent1_features, agent2_features, agent3_features, feature_categories = split_features_by_agent_definitions(
    all_features, agent_definitions
)

label_col = "label"   # binary 0/1

agent_names = agent_definitions["agent_names"]
agent_domains = agent_definitions["agent_domains"]
agent_actions = [
    ", ".join(agent_definitions["agent_actions"][i]) if agent_definitions["agent_actions"][i] else ""
    for i in range(3)
]
agent_feature_groups = [
    f"Features: {', '.join(agent1_features[:3])}..." if len(agent1_features) > 3 else f"Features: {', '.join(agent1_features)}",
    f"Features: {', '.join(agent2_features[:3])}..." if len(agent2_features) > 3 else f"Features: {', '.join(agent2_features)}",
    f"Features: {', '.join(agent3_features[:3])}..." if len(agent3_features) > 3 else f"Features: {', '.join(agent3_features)}"
]

# -----------------------------


In [4]:
# 3. Preprocess Features
# -----------------------------
scaler1 = StandardScaler()
scaler2 = StandardScaler()
scaler3 = StandardScaler()

X1 = torch.tensor(scaler1.fit_transform(df[agent1_features].values), dtype=torch.float32)
X2 = torch.tensor(scaler2.fit_transform(df[agent2_features].values), dtype=torch.float32)
X3 = torch.tensor(scaler3.fit_transform(df[agent3_features].values), dtype=torch.float32)
y = torch.tensor(df[label_col].values, dtype=torch.float32)

x_parts = [X1, X2, X3]

# -----------------------------


In [5]:
# 4. Train/Test Split
# -----------------------------
train_idx, test_idx = train_test_split(
    range(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=df[label_col]
)

x_train_parts = [x[train_idx] for x in x_parts]
x_test_parts = [x[test_idx] for x in x_parts]
y_train = y[train_idx]
y_test = y[test_idx]

# -----------------------------


In [6]:
# 5. VFL Model Definition
# -----------------------------
class LocalEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class ActiveClassifier(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, h):
        return torch.sigmoid(self.fc(h))


class VFLModel(nn.Module):
    def __init__(self, input_dims, embed_dim=32):
        super().__init__()
        self.embed_dim = embed_dim
        self.encoders = nn.ModuleList([LocalEncoder(dim, embed_dim) for dim in input_dims])
        self.classifier = ActiveClassifier(embed_dim)

    def forward(self, x_parts):
        embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]  # list of [B, D]
        h = torch.stack(embeddings).sum(dim=0)                           # [B, D]
        y_hat = self.classifier(h)                                       # [B, 1]
        return y_hat

    def get_agent_embeddings(self, x_parts):
        self.eval()
        with torch.no_grad():
            embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]
        return embeddings  # list: [B, D] for each agent

# -----------------------------


In [7]:
# 6. Train & Evaluate Functions
# -----------------------------
def train_vfl(model, x_parts, y, epochs=100, lr=1e-3):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        y_hat = model(x_parts).squeeze()
        loss = criterion(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            print(f"[VFL] Epoch {epoch} | Loss: {loss.item():.4f}")

def evaluate(model, x_parts, y):
    model.eval()
    with torch.no_grad():
        y_hat = model(x_parts).squeeze()
        y_pred = (y_hat > 0.5).int()
    acc = accuracy_score(y.cpu(), y_pred.cpu())
    rec = recall_score(y.cpu(), y_pred.cpu())
    f1 = f1_score(y.cpu(), y_pred.cpu())
    print(f"[VFL] Accuracy: {acc:.4f} | Recall: {rec:.4f} | F1-score: {f1:.4f}")
    return acc, rec, f1

# -----------------------------


In [8]:
# 7. Train VFL Model
# -----------------------------
embed_dim = 32
model = VFLModel(
    input_dims=[len(agent1_features), len(agent2_features), len(agent3_features)],
    embed_dim=embed_dim
)

train_vfl(model, x_train_parts, y_train, epochs=100)
acc, rec, f1 = evaluate(model, x_test_parts, y_test)

perf_df = pd.DataFrame({"Metric": ["Accuracy", "Recall", "F1"], "Value": [acc, rec, f1]})
perf_df.to_csv("vfl_shap_performance.csv", index=False)

# -----------------------------


[VFL] Epoch 0 | Loss: 0.6488
[VFL] Epoch 10 | Loss: 0.5106
[VFL] Epoch 20 | Loss: 0.3993
[VFL] Epoch 30 | Loss: 0.3115
[VFL] Epoch 40 | Loss: 0.2429
[VFL] Epoch 50 | Loss: 0.1893
[VFL] Epoch 60 | Loss: 0.1485
[VFL] Epoch 70 | Loss: 0.1175
[VFL] Epoch 80 | Loss: 0.0938
[VFL] Epoch 90 | Loss: 0.0755
[VFL] Accuracy: 1.0000 | Recall: 1.0000 | F1-score: 1.0000


In [9]:
# 8. Build Agent-Level Meta-Features
# -----------------------------
model.eval()
with torch.no_grad():
    train_embeds = model.get_agent_embeddings(x_train_parts)  # list [N_train, D]
    test_embeds = model.get_agent_embeddings(x_test_parts)    # list [N_test, D]

h1_train = train_embeds[0].mean(dim=1, keepdim=True)
h2_train = train_embeds[1].mean(dim=1, keepdim=True)
h3_train = train_embeds[2].mean(dim=1, keepdim=True)

h1_test = test_embeds[0].mean(dim=1, keepdim=True)
h2_test = test_embeds[1].mean(dim=1, keepdim=True)
h3_test = test_embeds[2].mean(dim=1, keepdim=True)

X_train_meta = torch.cat([h1_train, h2_train, h3_train], dim=1)  # [N_train, 3]
X_test_meta  = torch.cat([h1_test,  h2_test,  h3_test],  dim=1)  # [N_test, 3]

# -----------------------------


In [10]:
# 9. Meta-Model on [h1, h2, h3]
# -----------------------------
class AgentMetaModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3, 1)
    def forward(self, x_meta):
        return torch.sigmoid(self.fc(x_meta))

meta_model = AgentMetaModel()
criterion = nn.BCELoss()
optimizer = optim.Adam(meta_model.parameters(), lr=1e-3)
meta_epochs = 50

model.eval()
for epoch in range(meta_epochs):
    meta_model.train()
    optimizer.zero_grad()
    with torch.no_grad():
        y_hat_teacher = model(x_train_parts).squeeze()
    y_hat_meta = meta_model(X_train_meta).squeeze()
    loss = criterion(y_hat_meta, y_hat_teacher)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"[META] Epoch {epoch} | Distillation Loss: {loss.item():.4f}")

# Meta-model fidelity
meta_model.eval()
with torch.no_grad():
    y_hat_teacher_test = model(x_test_parts).squeeze().cpu().numpy()
    y_hat_meta_test = meta_model(X_test_meta).squeeze().cpu().numpy()
meta_corr = np.corrcoef(y_hat_teacher_test, y_hat_meta_test)[0, 1]
print(f"[META] Correlation between VFL and meta-model outputs on test set: {meta_corr:.4f}")

# -----------------------------


[META] Epoch 0 | Distillation Loss: 0.6859
[META] Epoch 10 | Distillation Loss: 0.6836
[META] Epoch 20 | Distillation Loss: 0.6815
[META] Epoch 30 | Distillation Loss: 0.6794
[META] Epoch 40 | Distillation Loss: 0.6774
[META] Correlation between VFL and meta-model outputs on test set: 1.0000


In [11]:
# 10. SHAP on Agent Meta-Features (KernelExplainer)
# -----------------------------
meta_model.eval()

bg_size = min(100, X_train_meta.shape[0])
bg_idx = torch.randperm(X_train_meta.shape[0])[:bg_size]
background = X_train_meta[bg_idx].detach().cpu().numpy()  # [bg_size, 3]

def meta_predict(x_np):
    x_t = torch.tensor(x_np, dtype=torch.float32)
    with torch.no_grad():
        out = meta_model(x_t).detach().cpu().numpy().reshape(-1)
    return out

explainer = shap.KernelExplainer(meta_predict, background)

n_explain = min(200, X_test_meta.shape[0])
X_explain = X_test_meta[:n_explain].detach().cpu().numpy()  # [n_explain, 3]

shap_values = explainer.shap_values(X_explain, nsamples=200)
shap_values = np.asarray(shap_values)  # [n_explain, 3]

# -----------------------------


  0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# 11. Aggregate SHAP for tables
# -----------------------------
phi = shap_values  # [n_explain, 3]
phi_abs = np.abs(phi)

# Global mean |SHAP| and percentage per party
mean_phi_abs = phi_abs.mean(axis=0)              # [3]
mean_phi_pct = mean_phi_abs / mean_phi_abs.sum() # [3]

print("\n=== Global SHAP Agent Attributions (All Explained Flows) ===")
global_rows = []
for i, name in enumerate(party_names):
    m_abs = float(np.asarray(mean_phi_abs[i]))
    m_pct = float(np.asarray(mean_phi_pct[i]))
    print(f"{name}: mean |SHAP| = {m_abs:.6f}, mean contribution = {m_pct*100:5.2f}%")
    global_rows.append({
        "Party": name,
        "Domain": agent_domains[i],
        "Feature_Group": agent_feature_groups[i],
        "Mean_abs_SHAP_All": m_abs,
        "Mean_contrib_All": m_pct,
    })
global_df = pd.DataFrame(global_rows)
global_df.to_csv("vfl_shap_global_summary.csv", index=False)

# Class-conditional SHAP (DDoS vs benign) on explained subset
y_test_np = y_test.cpu().numpy()
y_explain = y_test_np[:n_explain]

attack_mask = (y_explain == 1)
benign_mask = (y_explain == 0)

def compute_subset_shap(mask):
    if mask.sum() == 0:
        return np.full(3, np.nan), np.full(3, np.nan)
    phi_sub = phi_abs[mask]
    mean_phi_sub = phi_sub.mean(axis=0)
    mean_pct_sub = mean_phi_sub / mean_phi_sub.sum()
    return mean_phi_sub, mean_pct_sub

mean_phi_attack, mean_pct_attack = compute_subset_shap(attack_mask)
mean_phi_benign, mean_pct_benign = compute_subset_shap(benign_mask)

print("\n=== SHAP Agent Attributions (DDoS/attack flows) ===")
attack_rows = []
for i, name in enumerate(party_names):
    a_abs = float(np.asarray(mean_phi_attack[i]))
    a_pct = float(np.asarray(mean_pct_attack[i]))
    print(f"{name}: mean |SHAP| = {a_abs:.6f}, mean contribution = {a_pct*100:5.2f}%")
    attack_rows.append({
        "Party": name,
        "Domain": agent_domains[i],
        "Feature_Group": agent_feature_groups[i],
        "Mean_abs_SHAP_Attack": a_abs,
        "Mean_contrib_Attack": a_pct,
    })
attack_df = pd.DataFrame(attack_rows)
attack_df.to_csv("vfl_shap_ddos_summary.csv", index=False)

print("\n=== SHAP Agent Attributions (Benign flows) ===")
benign_rows = []
for i, name in enumerate(party_names):
    b_abs = float(np.asarray(mean_phi_benign[i]))
    b_pct = float(np.asarray(mean_pct_benign[i]))
    print(f"{name}: mean |SHAP| = {b_abs:.6f}, mean contribution = {b_pct*100:5.2f}%")
    benign_rows.append({
        "Party": name,
        "Domain": agent_domains[i],
        "Feature_Group": agent_feature_groups[i],
        "Mean_abs_SHAP_Benign": b_abs,
        "Mean_contrib_Benign": b_pct,
    })
benign_df = pd.DataFrame(benign_rows)
benign_df.to_csv("vfl_shap_benign_summary.csv", index=False)

# -----------------------------



=== Global SHAP Agent Attributions (All Explained Flows) ===
Agent 1: mean |SHAP| = 0.006103, mean contribution = 23.05%
Agent 2: mean |SHAP| = 0.013483, mean contribution = 50.93%
Agent 3: mean |SHAP| = 0.006889, mean contribution = 26.02%

=== SHAP Agent Attributions (DDoS/attack flows) ===
Agent 1: mean |SHAP| = 0.003774, mean contribution = 11.51%
Agent 2: mean |SHAP| = 0.024318, mean contribution = 74.16%
Agent 3: mean |SHAP| = 0.004698, mean contribution = 14.33%

=== SHAP Agent Attributions (Benign flows) ===
Agent 1: mean |SHAP| = 0.008431, mean contribution = 41.82%
Agent 2: mean |SHAP| = 0.002648, mean contribution = 13.14%
Agent 3: mean |SHAP| = 0.009079, mean contribution = 45.04%


In [13]:
# 11b. Agent-level mitigation summary
# -----------------------------
top_party_idx = int(np.nanargmax(mean_pct_attack))
top_party_name = party_names[top_party_idx]
top_party_domain = agent_domains[top_party_idx]
top_party_action = agent_actions[top_party_idx]
top_party_share = float(mean_pct_attack[top_party_idx]) * 100.0

print("\n=== Agentic Mitigation Recommendation (Attack flows) ===")
print(f"Dominant party for DDoS decisions: {top_party_name} ({top_party_domain}) "
      f"with mean SHAP contribution ≈ {top_party_share:5.2f}%.")

print(f"Recommended primary mitigation: {top_party_action}.")

agent_rows = []
for i, name in enumerate(party_names):
    agent_rows.append({
        "Party": name,
        "Domain": agent_domains[i],
        "Feature_Group": agent_feature_groups[i],
        "Mean_contrib_Attack": float(mean_pct_attack[i]),
        "Suggested_Action": agent_actions[i]
    })
agent_df = pd.DataFrame(agent_rows)
agent_df.to_csv("vfl_shap_agent_mitigation_summary.csv", index=False)

# -----------------------------



=== Agentic Mitigation Recommendation (Attack flows) ===
Dominant party for DDoS decisions: Party 2 (Cloud / datacenter) with mean SHAP contribution ≈ 74.16%.
Recommended primary mitigation: reconfigure load balancer / autoscale VNFs.


In [14]:
# 12. Example per-flow SHAP with agent hints
# -----------------------------
print("\n=== Example per-flow party SHAP (first 5 explained flows) ===")
example_rows = []
for i in range(min(5, n_explain)):
    phi_i = np.asarray(phi[i]).reshape(-1)
    phi_abs_i = np.abs(phi_i)
    if phi_abs_i.sum() == 0:
        pct_i = np.zeros_like(phi_abs_i)
    else:
        pct_i = phi_abs_i / phi_abs_i.sum()

    label_i = int(y_explain[i])
    print(f"\nFlow {i} (label = {label_i}):")
    row = {
        "FlowIndex": i,
        "Label": label_i,
    }

    for j, name in enumerate(party_names):
        phi_j = float(phi_i[j])
        pct_j = float(pct_i[j])
        print(f"  {name} SHAP: {phi_j: .4f} | contribution: {pct_j*100:5.2f}%")
        row[f"{name}_SHAP"] = phi_j
        row[f"{name}_contrib"] = pct_j

    if label_i == 1:
        top_j = int(np.argmax(pct_i))
        row["Top_Party"] = party_names[top_j]
        row["Top_Domain"] = agent_domains[top_j]
        row["Suggested_Action"] = agent_actions[top_j]
        print(f"  -> Agent recommendation: {party_names[top_j]} "
              f"({agent_domains[top_j]}) should {agent_actions[top_j]}.")
    else:
        row["Top_Party"] = ""
        row["Top_Domain"] = ""
        row["Suggested_Action"] = ""

    example_rows.append(row)

examples_df = pd.DataFrame(example_rows)
examples_df.to_csv("vfl_shap_flow_examples.csv", index=False)



=== Example per-flow party SHAP (first 5 explained flows) ===

Flow 0 (label = 1):
  Party 1 SHAP:  0.0038 | contribution: 11.51%
  Party 2 SHAP:  0.0243 | contribution: 74.16%
  Party 3 SHAP:  0.0047 | contribution: 14.33%
  -> Agent recommendation: Party 2 (Cloud / datacenter) should reconfigure load balancer / autoscale VNFs.

Flow 1 (label = 0):
  Party 1 SHAP: -0.0084 | contribution: 41.82%
  Party 2 SHAP: -0.0026 | contribution: 13.14%
  Party 3 SHAP: -0.0091 | contribution: 45.04%


In [15]:
# -----------------------------
# 13. Simulated mitigation policies (for magazine table)
# -----------------------------
# We use:
#  - y_hat_teacher_test: VFL predicted probabilities on test set (already computed above)
#  - y_test_np: true labels on test set
#  - phi, y_explain: SHAP values and labels for first n_explain test samples

print("\n=== Simulated Mitigation Policies ===")

# Ensure numpy arrays
y_scores = y_hat_teacher_test           # shape [N_test]
y_true   = y_test_np                    # shape [N_test]

# --- Policy 1: Global threshold only ---
# Block any flow with predicted DDoS probability > 0.5
blocked_global = (y_scores > 0.5)

# Attack = 1, Benign = 0
attack_mask_full = (y_true == 1)
benign_mask_full = (y_true == 0)

attack_block_rate_global = (
    blocked_global[attack_mask_full].mean() if attack_mask_full.sum() > 0 else np.nan
)
benign_block_rate_global = (
    blocked_global[benign_mask_full].mean() if benign_mask_full.sum() > 0 else np.nan
)

# --- Policy 2: Agentic SHAP-guided mitigation ---
# Use only first n_explain flows where we have SHAP (phi, y_explain)
# For each explained flow:
#   - If predicted as attack (y_scores > 0.5) AND true label = 1
#   - AND Party 3 has highest SHAP share and share > 0.7
#   -> blocked by enterprise agent.
#
# For benign flows, we check how many would be (incorrectly) blocked by same rule.

n_explain = min(n_explain, len(y_scores))
scores_explain = y_scores[:n_explain]
labels_explain = y_explain[:n_explain]   # already defined above for SHAP subsets

# Compute per-flow SHAP contributions
phi_explain = phi[:n_explain]            # [n_explain, 3]
phi_abs_explain = np.abs(phi_explain)
phi_sum = phi_abs_explain.sum(axis=1, keepdims=True)
phi_sum[phi_sum == 0] = 1.0
pct_explain = phi_abs_explain / phi_sum  # SHAP share per party

# Conditions for agentic blocking
pred_attack = (scores_explain > 0.5)
true_attack = (labels_explain == 1)
true_benign = (labels_explain == 0)

top_party = np.argmax(pct_explain, axis=1)          # index of dominant party
top_share = pct_explain[np.arange(n_explain), top_party]

# Agentic rule: block if predicted attack AND Party 3 dominant AND share > 0.7
agent_block = (pred_attack & (top_party == 2) & (top_share > 0.7))

# Compute rates over explained subset
attack_mask_explain = true_attack
benign_mask_explain = true_benign

attack_block_rate_agent = (
    agent_block[attack_mask_explain].mean() if attack_mask_explain.sum() > 0 else np.nan
)
benign_block_rate_agent = (
    agent_block[benign_mask_explain].mean() if benign_mask_explain.sum() > 0 else np.nan
)

# Convert to percentages
attack_block_rate_global_pct = float(attack_block_rate_global * 100.0)
benign_block_rate_global_pct = float(benign_block_rate_global * 100.0)
attack_block_rate_agent_pct  = float(attack_block_rate_agent * 100.0)
benign_block_rate_agent_pct  = float(benign_block_rate_agent * 100.0)

print("\nPolicy\t\t\tAttack blocked (%)\tBenign blocked (%)")
print(f"Global threshold only\t{attack_block_rate_global_pct:6.2f}\t\t\t{benign_block_rate_global_pct:6.2f}")
print(f"Agentic SHAP-guided\t{attack_block_rate_agent_pct:6.2f}\t\t\t{benign_block_rate_agent_pct:6.2f}")

# Save as CSV for magazine table
policy_rows = [
    {
        "Policy": "Global threshold only",
        "Attack_blocked_pct": attack_block_rate_global_pct,
        "Benign_blocked_pct": benign_block_rate_global_pct,
    },
    {
        "Policy": "Agentic SHAP-guided mitigation",
        "Attack_blocked_pct": attack_block_rate_agent_pct,
        "Benign_blocked_pct": benign_block_rate_agent_pct,
    }
]

policy_df = pd.DataFrame(policy_rows)
policy_df.to_csv("vfl_shap_mitigation_policies.csv", index=False)



=== Simulated Mitigation Policies ===

Policy			Attack blocked (%)	Benign blocked (%)
Global threshold only	100.00			  0.00
Agentic SHAP-guided	  0.00			  0.00
